# Task 1: Collaborative Filtering — Memory-Based vs Matrix Factorization

This notebook extends the existing MovieLens 100K collaborative-filtering implementation. It keeps the same dataset, preprocessing, chronological 80/20 train-test split, and item-item KNN approach, then adds **matrix factorization from scratch** using SGD.

Both methods are evaluated on the **same held-out test interactions** using RMSE and MAE, and both can generate top-N recommendations.

## 1. Objective

We compare:

- **Memory-based CF:** item-item KNN using cosine/Pearson-style mean-centered similarity computed directly from the training interaction matrix.
- **Model-based CF:** matrix factorization, learning compact user and item latent vectors from observed ratings.

The key comparison is fair because both models use the same MovieLens data and the same chronological train/test split.

## 2. Dataset and preprocessing

MovieLens 100K is appropriate because it contains explicit user ratings for movies, giving us a natural user-item interaction matrix. The supplied advertising dataset is not needed for this task.

Important missing-value rule:

- Missing values in the raw `rating` column are invalid records and are removed.
- `NaN` values inside the user-item matrix mean **the user has not rated that movie**. They are not converted to zero.
- Matrix factorization trains only on observed `(user, item, rating)` triples.
- Test users/items not present in training are excluded from the basic evaluation because they represent the cold-start case.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Reuse the MovieLens 100K source used in the original notebook
url = "https://files.grouplens.org/datasets/movielens/ml-100k/u.data"
columns = ['userId', 'movieId', 'rating', 'timestamp']
ratings = pd.read_csv(url, sep='\t', names=columns)

print("Shape:", ratings.shape)
print(ratings.head())
print("\nMissing values:\n", ratings.isnull().sum())

In [ ]:
# Basic preprocessing
ratings = ratings.dropna(subset=['userId', 'movieId', 'rating']).copy()

# Check duplicate user-item interactions.
duplicates = ratings.duplicated(subset=['userId', 'movieId']).sum()
print("Duplicate user-item pairs:", duplicates)

# MovieLens 100K normally has no duplicate user-item pairs.
# If duplicates exist, aggregate them so every user-item pair has one rating.
if duplicates > 0:
    ratings = (ratings.groupby(['userId', 'movieId'], as_index=False)
                      .agg({'rating': 'mean', 'timestamp': 'max'}))

# Chronological split: the earliest 80% is training data and the latest 20% is test data.
ratings = ratings.sort_values('timestamp').reset_index(drop=True)
split = int(len(ratings) * 0.80)

train = ratings.iloc[:split].copy()
test = ratings.iloc[split:].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train time range:", train['timestamp'].min(), "to", train['timestamp'].max())
print("Test time range:", test['timestamp'].min(), "to", test['timestamp'].max())

## 3. Memory-based CF: item-item KNN

This is the existing approach from the original notebook. We create a training-only user-item matrix. Its `NaN`s mean that a rating was not observed.

For item similarity, the notebook mean-centers item vectors and computes their normalized dot product. For a target user and movie, the prediction is a similarity-weighted average of the user's ratings on the most similar movies.

In [ ]:
# Training-only interaction matrix
R_train = train.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

print("User-item matrix shape:", R_train.shape)
print("Number of unobserved cells:", int(R_train.isna().sum().sum()))

In [ ]:
# Mean-center item vectors for similarity calculation.
# Filling with item means is ONLY for computing the similarity matrix;
# the original NaNs are retained in R_train for prediction logic.
movie_means = R_train.mean(axis=0)
R_train_filled = R_train.fillna(movie_means).fillna(0)

X = R_train_filled.to_numpy().T  # rows = movies, columns = users
item_means = np.mean(X, axis=1, keepdims=True)
X_centered = X - item_means

numerator = np.dot(X_centered, X_centered.T)
sum_of_squares = np.sum(X_centered ** 2, axis=1)
denominator = np.sqrt(np.outer(sum_of_squares, sum_of_squares))

similarity = np.divide(
    numerator,
    denominator,
    out=np.zeros_like(numerator),
    where=denominator != 0
)

sim_df = pd.DataFrame(similarity, index=R_train.columns, columns=R_train.columns)
print("Similarity matrix shape:", sim_df.shape)

In [ ]:
def predict_rating_knn(user_id, item_id, k=20):
    """Predict a rating with item-item KNN using only training interactions."""
    if user_id not in R_train.index or item_id not in R_train.columns:
        return np.nan

    item_similarities = sim_df[item_id]
    user_ratings = R_train.loc[user_id]

    # Only movies this user actually rated.
    rated_mask = user_ratings.notna()
    sims_for_rated = item_similarities[rated_mask]
    ratings_for_rated = user_ratings[rated_mask]

    # Ignore negative similarities for this weighted-average predictor.
    positive_mask = sims_for_rated > 0
    sims_for_rated = sims_for_rated[positive_mask]
    ratings_for_rated = ratings_for_rated[positive_mask]

    # The target movie itself may be among rated movies during evaluation.
    # Remove it so the model cannot directly reuse the answer.
    if item_id in sims_for_rated.index:
        sims_for_rated = sims_for_rated.drop(index=item_id)
        ratings_for_rated = ratings_for_rated.drop(index=item_id)

    if len(sims_for_rated) == 0:
        return float(train['rating'].mean())

    top_k_sims = sims_for_rated.nlargest(k)
    top_k_ratings = ratings_for_rated.loc[top_k_sims.index]

    if top_k_sims.sum() == 0:
        return float(train['rating'].mean())

    prediction = np.dot(top_k_sims.to_numpy(), top_k_ratings.to_numpy()) / top_k_sims.sum()
    return float(np.clip(prediction, 1, 5))

In [ ]:
def evaluate_knn(test_df, k=20):
    actuals, predictions = [], []

    for _, row in test_df.iterrows():
        user, item, actual = row['userId'], row['movieId'], row['rating']
        if user in R_train.index and item in R_train.columns:
            pred = predict_rating_knn(user, item, k=k)
            if not np.isnan(pred):
                actuals.append(actual)
                predictions.append(pred)

    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    mae = mean_absolute_error(actuals, predictions)
    coverage = len(predictions) / len(test_df)
    return rmse, mae, coverage, np.array(actuals), np.array(predictions)

knn_rmse, knn_mae, knn_coverage, knn_actual, knn_pred = evaluate_knn(test, k=20)
print(f"Item-item KNN RMSE: {knn_rmse:.4f}")
print(f"Item-item KNN MAE : {knn_mae:.4f}")
print(f"Evaluation coverage: {knn_coverage:.2%}")

In [ ]:
def recommend_movies_knn(user_id, n=10, k=20):
    """Return top-N unseen movies predicted by item-item KNN."""
    if user_id not in R_train.index:
        return []

    user_ratings = R_train.loc[user_id]
    unrated_movies = user_ratings[user_ratings.isna()].index

    predictions = []
    for movie_id in unrated_movies:
        pred = predict_rating_knn(user_id, movie_id, k=k)
        if not np.isnan(pred):
            predictions.append((movie_id, pred))

    predictions.sort(key=lambda x: x[1], reverse=True)
    return predictions[:n]

sample_user = R_train.index[0]
print("Sample KNN recommendations:")
print(recommend_movies_knn(sample_user, n=10, k=20))

## 4. Model-based CF: Matrix Factorization from scratch

We approximate the rating matrix using latent user and item factors. We use the biased formulation:

\[
\hat r_{ui}=\mu+b_u+b_i+U_u^T V_i
\]

where `mu` is the global mean, `b_u` and `b_i` are user/item biases, and `U_u`, `V_i` are latent vectors.

The model is trained only on observed training ratings:

\[
L=\sum_{(u,i)\in\Omega}(r_{ui}-\hat r_{ui})^2 + \lambda(\|U\|^2+\|V\|^2+\|b_u\|^2+\|b_i\|^2)
\]

This is important: the `NaN`s in `R_train` are **not** treated as zero ratings and are never included in the loss.

In [ ]:
# Create integer indices using TRAINING data only.
user_ids = train['userId'].unique()
movie_ids = train['movieId'].unique()

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_to_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

train_mf = train.copy()
test_mf = test.copy()

train_mf['user_idx'] = train_mf['userId'].map(user_to_idx)
train_mf['movie_idx'] = train_mf['movieId'].map(movie_to_idx)

test_mf['user_idx'] = test_mf['userId'].map(user_to_idx)
test_mf['movie_idx'] = test_mf['movieId'].map(movie_to_idx)

# Unseen test users/items are the cold-start case.
test_mf_eval = test_mf.dropna(subset=['user_idx', 'movie_idx']).copy()

test_mf_eval['user_idx'] = test_mf_eval['user_idx'].astype(int)
test_mf_eval['movie_idx'] = test_mf_eval['movie_idx'].astype(int)

train_users = train_mf['user_idx'].astype(int).to_numpy()
train_movies = train_mf['movie_idx'].astype(int).to_numpy()
train_ratings = train_mf['rating'].astype(float).to_numpy()

test_users = test_mf_eval['user_idx'].to_numpy()
test_movies = test_mf_eval['movie_idx'].to_numpy()
test_ratings = test_mf_eval['rating'].astype(float).to_numpy()

print("Training interactions:", len(train_ratings))
print("Test interactions evaluable by MF:", len(test_ratings))
print("Cold-start test interactions excluded:", len(test_mf) - len(test_mf_eval))

In [ ]:
# Matrix factorization hyperparameters
np.random.seed(42)

num_users = len(user_ids)
num_movies = len(movie_ids)
latent_dim = 20
epochs = 30
learning_rate = 0.005
regularization = 0.02

global_mean = train_mf['rating'].mean()

# Latent factors and biases
U = np.random.normal(0, 0.1, size=(num_users, latent_dim))
V = np.random.normal(0, 0.1, size=(num_movies, latent_dim))
user_bias = np.zeros(num_users)
movie_bias = np.zeros(num_movies)

train_losses = []

for epoch in range(epochs):
    squared_error = 0.0
    indices = np.random.permutation(len(train_ratings))

    for idx in indices:
        u = train_users[idx]
        i = train_movies[idx]
        r = train_ratings[idx]

        prediction = global_mean + user_bias[u] + movie_bias[i] + np.dot(U[u], V[i])
        error = r - prediction
        squared_error += error ** 2

        # Copies prevent the second update from using a partially updated vector.
        user_vector = U[u].copy()
        movie_vector = V[i].copy()

        user_bias[u] += learning_rate * (error - regularization * user_bias[u])
        movie_bias[i] += learning_rate * (error - regularization * movie_bias[i])

        U[u] += learning_rate * (error * movie_vector - regularization * user_vector)
        V[i] += learning_rate * (error * user_vector - regularization * movie_vector)

    epoch_mse = squared_error / len(train_ratings)
    train_losses.append(epoch_mse)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:2d}/{epochs} - Training MSE: {epoch_mse:.4f}")

In [ ]:
def mf_predict(user_idx, movie_idx):
    pred = (
        global_mean
        + user_bias[user_idx]
        + movie_bias[movie_idx]
        + np.dot(U[user_idx], V[movie_idx])
    )
    return float(np.clip(pred, 1, 5))

mf_predictions = np.array([
    mf_predict(u, i) for u, i in zip(test_users, test_movies)
])

mf_rmse = np.sqrt(mean_squared_error(test_ratings, mf_predictions))
mf_mae = mean_absolute_error(test_ratings, mf_predictions)
mf_coverage = len(test_ratings) / len(test_mf)

print(f"Matrix Factorization RMSE: {mf_rmse:.4f}")
print(f"Matrix Factorization MAE : {mf_mae:.4f}")
print(f"Evaluation coverage: {mf_coverage:.2%}")

In [ ]:
# Plot matrix-factorization training convergence
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), train_losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Training MSE')
plt.title('Matrix Factorization Training Loss')
plt.grid(True)
plt.show()

## 5. Matrix-factorization recommendations

For recommendation, we predict every movie known during training that the selected user has not already rated. The movies are ranked by predicted rating and the highest-scoring items are returned.

In [ ]:
def recommend_movies_mf(user_id, n=10):
    """Return top-N unseen movies using the learned MF model."""
    if user_id not in user_to_idx:
        return []

    u = user_to_idx[user_id]
    rated_movies = set(train.loc[train['userId'] == user_id, 'movieId'])

    candidates = [movie_id for movie_id in movie_ids if movie_id not in rated_movies]
    predictions = []

    for movie_id in candidates:
        i = movie_to_idx[movie_id]
        predictions.append((movie_id, mf_predict(u, i)))

    predictions.sort(key=lambda x: x[1], reverse=True)
    return predictions[:n]

print("Sample MF recommendations:")
print(recommend_movies_mf(sample_user, n=10))

## 6. Direct comparison

Both methods now use the same MovieLens 100K dataset and the same chronological split. RMSE and MAE are computed on held-out ratings. Coverage is reported separately because memory-based prediction can fail to produce a usable neighborhood in sparse cases, while MF can predict any user-item pair whose user and item were seen during training.

In [ ]:
results = pd.DataFrame({
    'Method': ['Memory-based Item-Item KNN', 'Model-based Matrix Factorization'],
    'RMSE': [knn_rmse, mf_rmse],
    'MAE': [knn_mae, mf_mae],
    'Evaluation Coverage': [knn_coverage, mf_coverage]
})

print(results.to_string(index=False))

In [ ]:
# Comparison plot: lower is better for RMSE and MAE.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(results['Method'], results['RMSE'])
axes[0].set_title('RMSE Comparison')
axes[0].set_ylabel('RMSE')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(results['Method'], results['MAE'])
axes[1].set_title('MAE Comparison')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 7. Interpretation and conclusion

### Memory-based item-item KNN

Memory-based CF is simple and interpretable. It directly uses observed item-item relationships and is useful when similar historical interactions are plentiful. It does not require learning a latent representation. Its main limitations are sparse data, dependence on the similarity computation, and weaker generalization to interactions that do not have enough useful neighbors.

### Matrix factorization

Matrix factorization compresses users and movies into a low-dimensional latent space. Instead of comparing every pair of movies at prediction time, it learns factors that capture hidden preference patterns. This generally makes it more scalable and able to generalize beyond directly similar neighbors. Its trade-offs are the need to choose hyperparameters such as latent dimension, learning rate, regularization, and number of epochs, and the fact that completely new users/items remain a cold-start problem.

### Final comparison

The final choice should be based on the held-out RMSE/MAE and the practical requirements. If KNN gives lower error and the dataset is sufficiently dense, the memory-based method is a strong simple baseline. If matrix factorization gives lower error, it provides better predictive accuracy and a compact learned representation. For larger and sparser recommendation systems, model-based latent-factor methods are often preferable because they can generalize from many interactions without explicitly storing a large neighborhood calculation for every prediction.

**Important:** the numerical winner should be taken directly from the `results` table after running this notebook rather than assumed in advance.

## 8. Optional sanity checks

The following checks make the implementation easy to inspect during submission:

1. `R_train.isna()` represents unobserved interactions, not zero ratings.
2. Matrix factorization receives only observed training triples.
3. Test mappings are created from training IDs only, so cold-start interactions are visible rather than silently leaked into training.
4. Both algorithms evaluate on the same chronological test set.
5. No pretrained recommendation weights are used; the latent factors are initialized randomly and learned from the MovieLens training ratings.